# 04c · 재생 관점에서 본 마르코프 사슬 (Markov Chains from the Regenerative Viewpoint)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.5 (Limit Behavior: $N_n(y)/n\to1/E_yT_y$), §1.7 (Proofs: cycle trick) · Ross 12e §4.4 (Long-Run Proportions), §7.5 (Regenerative Processes) · Norris §1.4 (Strong Markov), §1.7 (Invariant distributions), §1.10 (Ergodic theorem) · Lawler 2e Ch. 1–2 |
| 선수 노트북 | 04a (재생 SLLN·재생-보상), 04b (격자 간격·시간평균), 01d (정상분포), 01e (수렴정리·주기), 01b (흡수·첫걸음 분석) |
| 예상 소요 | 90분 |
| 상태 | draft |

01d에서 정상분포 $\pi$를 "$\pi P=\pi$의 해"로, 01e에서 "$P^n$의 극한"으로 보았다. 이 노트북은 세 번째 얼굴을 보여 준다: **$\pi(x)$는 평균 복귀시간의 역수 $1/E_x[T_x]$이고, 이것은 04a의 재생 정리를 마르코프 사슬에 그대로 적용한 결과다.** 상태 $x$에 도장을 찍을 때마다 사슬은 "다시 태어나고"(강마르코프), 도장 사이의 조각들은 i.i.d.다. 그래서 비주기성 없이도 시간평균은 항상 수렴한다 — 04b에서 격자 간격이 극한분포는 깨뜨려도 시간평균은 남겨 두던 것과 같은 구조다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (stationary, stationary_eig, absorption, classify_states,
                          simulate_chain, empirical_distribution, ehrenfest_matrix,
                          random_walk_paths)
from spkit.plots import plot_hist_vs_pmf

SEED, rng = rng_for("04c")
setup_plots()
N_PATHS = scale(100, divisor=5, minimum=20)   # FAST(SPKIT_FAST=1) 모드에서는 20 경로
N_STEPS = 20_000
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_STEPS={N_STEPS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (04b) 평형분포 $F_e$의 정의와, Gamma(2,1) 간격에서 극한 나이의 평균은?

<details><summary>정답</summary>

$F_e(x)=\frac1\mu\int_0^x\bar F(y)\,dy$. 극한 나이의 평균은 $E[X^2]/(2\mu)=6/4=1.5$.

</details>

**Q2.** (04b) 검사 역설: 시각 $t$를 덮는 간격 길이의 극한 평균과 $\mu$의 관계, 등호 조건은?

<details><summary>정답</summary>

$E[L]=E[X^2]/\mu=\mu+\sigma^2/\mu\ge\mu$. 등호는 $\sigma^2=0$, 즉 간격이 상수일 때만.

</details>

**Q3.** (04b) 간격 분포가 격자(lattice)이면 무엇이 깨지고 무엇이 남는가?

<details><summary>정답</summary>

$t\to\infty$ **극한분포**가 깨진다(상수 간격 $d$면 $A(t)=t\bmod d$로 진동). **시간평균**(재생-보상)은 남아 여전히 $F_e$. 이 노트북의 주기적 사슬(Ehrenfest)이 정확히 같은 구조다: $P^n(0,0)$은 진동하지만 시간비율은 수렴한다.

</details>

**Q4.** (04a) 재생-보상 정리의 결론과 '무엇이 i.i.d.여야 하는가'?

<details><summary>정답</summary>

$R(t)/t\to E[R]/E[X]$ a.s. 쌍 $(X_i,R_i)$가 주기끼리 i.i.d.여야 한다(한 주기 안에서 $R_i$가 $X_i$에 의존하는 것은 허용). 이 노트북에서는 (사이클 길이 $T_x$, 사이클 안의 $y$ 방문 수 $V_y$)가 그 쌍이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **첫 복귀시간** $T_x=\min\{n\ge1: X_n=x\}$. $X_0=x$일 때 '복귀'이고, $X_0=y\ne x$이면 같은 정의가 첫 **도달시간**(hitting time)이며 $E_y[T_x]$로 쓴다. $k$번째 복귀 $T_x^{(k)}$, $T_x^{(0)}=0$.
- **재귀**(recurrent): $P_x(T_x<\infty)=1$. **양재귀**(positive recurrent): $E_x[T_x]<\infty$. **영재귀**(null recurrent): 재귀이지만 $E_x[T_x]=\infty$. 유한 기약 사슬은 항상 양재귀.
- **방문 횟수** $N_n(y)=\sum_{k=1}^n\mathbf 1\{X_k=y\}$. **사이클당 방문 수** $V_y=\sum_{n=0}^{T_x-1}\mathbf 1\{X_n=y\}$ — 출발 시각 0은 포함, 복귀 시각 $T_x$는 제외. 따라서 $V_x=1$이고 $\sum_yV_y=T_x$.
- **강마르코프 성질**(strong Markov property): 정지시간 $\tau<\infty$에 대해 $(X_{\tau+n})_{n\ge0}$는 $\mathcal F_\tau$가 주어졌을 때 $X_\tau$에서 출발한 사슬과 같은 법칙을 갖고 $\mathcal F_\tau$와 독립이다.
- **재생 과정**(regenerative process): 시각열 $0\le\tau_1<\tau_2<\cdots$에서 과정이 '다시 시작'하여 사이클이 i.i.d.인 과정. 마르코프 사슬은 $\tau_k=T_x^{(k)}$로 재생 과정이다.

**예제 사슬** (이 노트북 내내 사용)

$$P=\begin{pmatrix}1/2&1/2&0\\0&1/3&2/3\\3/4&0&1/4\end{pmatrix}$$

기약, 비주기(자기고리), 비가역(순환 흐름 $0\to1\to2\to0$). $\pi=(12,9,8)/29$이므로 $E_0T_0=29/12$, $E_1T_1=29/9$, $E_2T_2=29/8$ — 이 값들을 4절에서 두 가지 방법으로 정확히 계산하고 시뮬레이션으로 확인한다.

### 2.2 정리 1 — 복귀 사이클의 i.i.d.성 (강마르코프; Norris §1.4, Durrett 3e §1.7)

기약 재귀 사슬에서 $x$로의 복귀 시각 차 $T_x^{(k)}-T_x^{(k-1)}$ ($k\ge1$)은 $P_x$ 아래 $T_x$와 같은 분포를 갖는 i.i.d.이며, 사이클 경로 조각 $(X_{T_x^{(k-1)}},\dots,X_{T_x^{(k)}-1})$들도 i.i.d.이다. $X_0=y\ne x$이면 첫 조각만 다른 분포(지연 재생 과정, delayed renewal process).

가정이 왜 필요한가:
- **강마르코프**: $T_x^{(k)}$가 정지시간이고 그 시각의 상태가 항상 $x$이므로, 이후는 $x$에서 새로 출발한 사슬이며 과거(앞선 사이클들)와 독립.
- **재귀**(기약이면 모든 상태가 재귀): 모든 $T_x^{(k)}<\infty$ a.s.여야 사이클이 무한히 많이 생긴다.

*증명 스케치.* 정지시간 $\tau=T_x^{(k)}$에 대해 $\{\tau=n\}$은 $X_0,\dots,X_n$만으로 결정된다. 그러므로 $\mathcal F_\tau$-가측 사건 $A$와 임의의 경로 사건 $B$에 대해
$P_x(A,\ \tau=n,\ (X_{\tau+m})_m\in B)=P_x(A,\tau=n)\,P_x(B)$ — 시각 $n$에서의 (보통) 마르코프 성질을 쓴 것이고 $X_n=x$이므로 우변에 $P_x(B)$가 나온다. $n$에 대해 더하면 $P_x(A,\ (X_{\tau+m})_m\in B)=P_x(A)P_x(B)$: 이것이 강마르코프다. $A$를 앞선 $k$개 사이클로 잡으면 $k+1$번째 사이클이 앞선 사이클들과 독립이고 첫 사이클과 같은 분포임이 나온다. 노트북에서는 연속한 복귀시간의 lag-1 자기상관 $\approx0$으로 확인한다. $\square$

### 2.3 정리 2 — 점근 빈도와 $\pi(x)=1/E_x[T_x]$ (Durrett 3e §1.5, Ross 12e §4.4)

$P$가 기약·양재귀(예: 유한 기약)이면 임의의 출발점에서

(i) $N_n(x)/n\to1/E_x[T_x]$ a.s.;
(ii) 유일한 정상분포 $\pi$는 $\pi(x)=1/E_x[T_x]$;
(iii) $E_x[V_y]=\pi(y)/\pi(x)$.

**비주기성은 필요 없다.**

가정이 왜 필요한가:
- **기약**: 정상분포의 유일성 + 어떤 출발점에서도 $x$에 a.s. 도달(지연 재생) + 극한이 출발점과 무관. 가약이면 시간비율이 출발점(어느 닫힌 류에 흡수되는가)에 의존한다.
- **양재귀** ($E_xT_x<\infty$): 04a 정리 1의 $\mu<\infty$. 영재귀면 극한이 0이라 (ii)가 '정상분포 없음'으로 바뀐다(E3, SRW).
- **비주기성이 필요 없는 이유**: 시간평균은 04b의 격자 논의처럼 주기성을 평균화한다. $P^n(x,x)\to\pi(x)$(01e)만 비주기성을 요구한다.

<details><summary>증명</summary>

세 줄이고, 각 줄이 쓰는 가정을 표시한다.

**1줄 (재생 SLLN; 가정: 강마르코프 + 재귀).** 정리 1에 의해 $x$로의 복귀 시각들은 간격이 i.i.d.이고 평균이 $\mu_x:=E_x[T_x]$인 재생 과정이다. 출발점이 $x$가 아니면 첫 도달시간 $T_x<\infty$ a.s.(기약·재귀)인 지연 재생 과정인데, 첫 조각 하나는 $n\to\infty$ 극한에 영향을 주지 못한다. 04a 정리 1(재생 SLLN — 이산시간에서도 샌드위치 $T_x^{(N_n(x))}\le n<T_x^{(N_n(x)+1)}$이 그대로 성립)에 의해

$$\frac{N_n(x)}n\to\frac1{\mu_x}\quad\text{a.s.}$$

이 줄만으로는 $\mu_x=\infty$도 허용되며 그때 극한은 0이다. 이것이 (i).

**2줄 (재생-보상; 가정: 양재귀).** 사이클 $k$의 보상을 $R_k=$ 그 사이클 동안의 $y$ 방문 수로 두자. 쌍 (사이클 길이, $R_k$)는 정리 1에 의해 사이클끼리 i.i.d.이고, $0\le R_k\le$ 사이클 길이이므로 $E|R_k|\le\mu_x<\infty$. 04a 정리 3(재생-보상)에 의해

$$\frac{N_n(y)}n\to\frac{E_x[V_y]}{E_x[T_x]}=:L(y)\quad\text{a.s.},$$

출발점과 무관하게. 여기서 $\mu_x<\infty$(양재귀)가 분모에 쓰였다.

**3줄 (정상 출발 + 유계수렴; 가정: 기약 → $\pi$ 존재·유일, 극한이 출발점 무관).** $X_0\sim\pi$로 출발시키자. 정상성에 의해 모든 $k$에서 $P_\pi(X_k=y)=\pi(y)$이므로 $E_\pi[N_n(y)/n]=\pi(y)$ (모든 $n$). $0\le N_n(y)/n\le1$이므로 유계수렴정리로

$$\pi(y)=\lim_n E_\pi\Big[\frac{N_n(y)}n\Big]=E_\pi\Big[\lim_n\frac{N_n(y)}n\Big]=L(y).$$

극한 $L(y)$가 출발점(여기서는 $\pi$에서 뽑힌 임의의 상태)과 무관하다는 2줄의 결론이 여기서 필요하다.

**결론.** $y=x$: $V_x=1$이므로 $\pi(x)=L(x)=1/E_x[T_x]$ — (ii). 일반 $y$: $\pi(y)=E_x[V_y]/E_x[T_x]=E_x[V_y]\,\pi(x)$ — (iii). $\square$

노트북에서는 (ii)를 `stationary(P)`와 첫걸음 분석(`absorption`)으로 정확히, (i)·(iii)를 시뮬레이션으로 확인한다.

</details>

### 2.4 정리 3 — 에르고딕 정리 (시간평균; Norris §1.10)

기약·양재귀이고 $f$가 유계(유한 상태면 임의의 $f$)이면

$$\frac1n\sum_{k=1}^nf(X_k)\to\sum_y\pi(y)f(y)\quad\text{a.s., 출발점 무관.}$$

가정이 왜 필요한가:
- **유계 $f$** (또는 $\sum_y\pi(y)|f(y)|<\infty$): 사이클 보상 $\sum_{\text{cycle}}f(X_n)$의 적분가능성.
- **기약·양재귀**: 정리 2와 동일. 비주기성 불필요.

*증명 스케치.* 정리 2의 재생-보상 논법에서 보상을 $R_k=\sum_{n\in\text{cycle }k}f(X_n)$으로 바꾸면 $|R_k|\le\|f\|_\infty\cdot$(사이클 길이)라 적분가능하고, 극한은
$E_x[\sum_{n<T_x}f(X_n)]/E_x[T_x]=\sum_yf(y)E_x[V_y]/E_x[T_x]=\sum_y\pi(y)f(y)$ (정리 2 (iii)). 유한 상태에서는 더 간단히 $f=\sum_yf(y)\mathbf 1_y$로 쓰고 정리 2 (i)을 유한 합해도 된다. 예제: $f(y)=y^2$이면 $\sum_y\pi(y)y^2=(0\cdot12+1\cdot9+4\cdot8)/29=41/29$. $\square$

### 2.5 정리 4 — 정상 측도의 사이클 구성 (cycle trick; Durrett 3e §1.7)

$x$가 재귀이면 $\mu_x(y):=E_x[V_y]$는 $\mu_xP=\mu_x$를 만족하는 **정상 측도**이고 총질량 $\sum_y\mu_x(y)=E_x[T_x]$이다. 양재귀면 정규화하여 $\pi=\mu_x/E_x[T_x]$.

가정이 왜 필요한가:
- **재귀만 필요(양재귀 아님)**: 영재귀에서도 정상 '측도'는 존재하지만 총질량이 무한이라 정규화가 안 된다 — SRW on $\mathbb Z$의 계수측도(모든 $y$에서 1)가 그 예(E3).

*증명 (인용; Durrett 3e §1.7).* $\mu_x(y)=\sum_{n\ge0}P_x(X_n=y,T_x>n)$이고 $\{T_x>n\}$은 $X_0,\dots,X_n$으로 결정되므로 마르코프 성질로 $\sum_y\mu_x(y)P(y,z)=\sum_{n\ge0}P_x(X_{n+1}=z,T_x>n)=\sum_{m\ge1}P_x(X_m=z,T_x\ge m)$. $z\ne x$면 이는 $\mu_x(z)$(시각 0 항은 0), $z=x$면 $P_x(T_x<\infty)=1=\mu_x(x)$. 정리 2 (iii)와 일치하며, 선형대수로는 $E_x[V_y]=\sum_{z\ne x}P(x,z)N_{zy}$, $N=(I-Q)^{-1}$ ($x$를 흡수 상태로 만든 기본행렬; E2).

### 2.6 직관

마르코프 사슬을 **'상태 $x$에 도장 찍기'**로 보자. 도장과 도장 사이의 길이 $T_x$가 i.i.d.이므로 도장 열은 재생 과정이고, 04a의 두 정리가 그대로 들어온다:

- 도장 빈도 $=1/E_x[T_x]$ (재생 SLLN),
- 사이클 안에서 벌어들인 것(다른 상태 방문, $f$의 합)의 장기 비율 $=$ 사이클당 기댓값$/E_x[T_x]$ (재생-보상).

그래서 $\pi$는 '평균 복귀시간의 역수'이고 $\pi(y)/\pi(x)$는 '한 $x$-사이클 동안 $y$를 몇 번 보는가'다. 01e의 수렴정리($P^n\to\pi$)가 비주기성을 요구하는 것과 달리 이 결과는 요구하지 않는다: 04b의 격자 간격처럼 주기성은 '시각 $n$의 분포'를 흔들지만 시간평균은 흔들지 못한다. Ehrenfest(공 4개)는 $P^n(0,0)$이 0과 $\approx1/8$을 번갈아 오가지만, 시간의 $1/16$을 0에서 보내고 $E_0T_0=16$이다.

### 2.7 함정 (traps)

- $E_x[T_x]$는 첫 **'복귀'**($n\ge1$) 시간이다. $T_x=1$(자기고리)도 포함되므로 예제 사슬에서 $P_0(T_0=1)=1/2$, $P_0(T_0=2)=0$ ($0\to1\to0$은 불가능), $P_0(T_0=3)=\frac12\cdot\frac23\cdot\frac34=\frac14$.
- $\pi(x)=1/E_x[T_x]$에는 비주기성이 필요 없다. 필요 없는 가정을 붙여 '주기적 사슬에는 $\pi$가 없다'고 오해하지 말 것 — 없는 것은 $\lim_nP^n$이다.
- 영재귀($E_xT_x=\infty$)에서는 복귀가 확실해도 $\pi(x)=0$: 정상분포가 없다. 시뮬레이션에서 관측된 복귀시간 평균은 지평이 길수록 커진다(E3).
- 가약 사슬에서는 시간비율이 출발점(어느 닫힌 류에 흡수되는가)에 따라 달라진다. 항등식은 각 닫힌 류 안에서 그 류의 $\pi$로 성립.
- 한 경로의 복귀시간들은 i.i.d.이므로 그냥 평균·SE를 써도 되지만, '한 경로의 시간비율'은 비율 추정량이라 SE는 경로 반복(또는 사이클 단위)으로 구해야 한다.
- `absorption`으로 $E_y[T_x]$를 구할 때는 $x$를 흡수 상태로 만든 $P_x$(행 $x$를 $e_x$로 교체)를 넣어야 한다; $E_x[T_x]=1+\sum_{y\ne x}P(x,y)E_y[T_x]$로 한 걸음 더 간다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — 특히 정리 2 증명의 세 줄(재생 SLLN → 재생-보상 → 정상 출발+유계수렴)을 각 줄이 어떤 가정을 쓰는지 표시하며, 그리고 '기약'을 빼면 어디서 무너지는지 한 문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 손계산 힌트: $\pi=(12,9,8)/29$, 첫걸음 분석, 그리고 Ehrenfest의 $\pi(k)=\binom4k/16$.

In [ ]:
predictions = {
    # 예제 사슬 P에서 0에서 출발해 0으로 처음 복귀하는 시간의 기댓값 E_0[T_0]은? (pi(0)=12/29)
    "E0_T0": None,
    # E_2[T_2]는?
    "E2_T2": None,
    # P_0(T_0 = 3)은? (P_0(T_0 = 2)는 얼마인지도 생각해 보세요)
    "P0_T0_eq_3": None,
    # 0에서 출발해 0으로 복귀하기 전까지 상태 1을 평균 몇 번 방문하는가? (E_0[V_1])
    "visits_1_per_cycle_from_0": None,
    # (1/n) sum_{k<=n} X_k^2 의 극한은?
    "ergodic_x2": None,
    # 공 4개 Ehrenfest 사슬(주기 2)에서 E_0[T_0]은?
    "ehrenfest_E0_T0": None,
}

## 4. Simulate — 시뮬레이션

먼저 이 노트북에서만 쓰는 작은 도우미들을 정의한다. 모두 numpy 몇 줄이며, 이름이 곧 정의다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def return_times(paths, x):
    """모든 경로의 x-복귀시간(연속한 x 방문 시각의 차)을 하나로 모은다. 첫 도달 이전 조각은 자동 제외."""
    return np.concatenate([np.diff(np.flatnonzero(row == x)) for row in paths])

def return_time_pmf(P, x, n_max):
    """P_x(T_x = n), n = 0..n_max. 시각 1은 자기고리, 시각 n>=2는 x 밖에서 n-2걸음 후 x로 진입."""
    n = len(P); T = [y for y in range(n) if y != x]
    Q, R, a = P[np.ix_(T, T)], P[T, x], P[x, T]
    pmf = np.zeros(n_max + 1); pmf[1] = P[x, x]; v = a.copy()
    for k in range(2, n_max + 1):
        pmf[k] = v @ R; v = v @ Q
    return pmf

def visits_per_cycle(path, x, y):
    """경로 하나에서 완결된 x-사이클마다 y 방문 수 (시각 0의 x 포함, 복귀 시각 제외)."""
    start = np.flatnonzero(path == x)[0]; p = path[start:]
    cyc = np.cumsum(p == x) - 1; n_cyc = cyc[-1]          # 마지막 미완 사이클 제외
    return np.bincount(cyc[p == y], minlength=n_cyc + 1)[:n_cyc]

def absorbing_at(P, x):
    """행 x를 e_x로 바꿔 x를 흡수 상태로 만든 사슬과 나머지 상태 목록."""
    P_x = P.copy(); P_x[x] = 0.0; P_x[x, x] = 1.0
    return P_x, [y for y in range(len(P)) if y != x]

def mean_return_time_exact(P, x):
    """E_x[T_x] = 1 + sum_{y != x} P(x, y) E_y[T_x]  (첫걸음 분석 + absorption)."""
    P_x, others = absorbing_at(P, x)
    return 1.0 + P[x, others] @ absorption(P_x, others, [x])["t"]

def expected_visits_exact(P, x, y):
    """E_x[V_y] = sum_{z != x} P(x, z) N_{zy},  N = (I - Q)^{-1}  (y != x)."""
    P_x, others = absorbing_at(P, x)
    return P[x, others] @ absorption(P_x, others, [x])["N"][:, others.index(y)]

def running_mean(a):
    return np.cumsum(a) / np.arange(1, len(a) + 1)

### 4.1 정확한 쪽: $\pi$와 $E_x[T_x]$를 두 방법으로

`stationary`(선형계 $\pi P=\pi$)와 첫걸음 분석(`absorption`)은 서로 전혀 다른 계산인데, 정리 2 (ii)는 그 곱이 정확히 1이어야 한다고 말한다.

In [ ]:
P = np.array([[.5, .5, 0.], [0., 1/3, 2/3], [.75, 0., .25]])
info = classify_states(P)
print("classes:", info["classes"], "| period:", info["period"], "| recurrent:", info["recurrent_states"])

pi = stationary(P)
print("pi           =", pi, " (x29 =", np.round(pi * 29, 10), ")")
print("stationary_eig =", stationary_eig(P))

ET = np.array([mean_return_time_exact(P, x) for x in range(3)])
lines = ["| x | pi(x) | E_x[T_x] (first-step) | 1/pi(x) | pi(x)·E_x[T_x] |", "|---|---|---|---|---|"]
for x in range(3):
    lines.append(f"| {x} | {pi[x]:.6f} | {ET[x]:.6f} | {1/pi[x]:.6f} | {pi[x]*ET[x]:.10f} |")
np.testing.assert_allclose(pi * ET, 1.0, atol=1e-10)   # 정확 항등식 pi(x) E_x[T_x] = 1
Markdown("\n".join(lines))

복귀시간의 **분포** 전체도 정확히 구할 수 있다: 시각 1의 복귀는 자기고리, 시각 $n\ge2$의 복귀는 "$x$ 밖에서 $n-2$걸음 머문 뒤 $x$로 진입"이라 $P_0(T_0=n)=a\,Q^{n-2}R$.

In [ ]:
N_MAX_PMF = 60
pmf0 = return_time_pmf(P, 0, N_MAX_PMF)
k = np.arange(N_MAX_PMF + 1)
mean0, sd0 = pmf0 @ k, np.sqrt(pmf0 @ k**2 - (pmf0 @ k) ** 2)
print("P_0(T_0 = n), n=1..6:", np.round(pmf0[1:7], 4), "| total mass:", pmf0.sum().round(10))
print(f"mean = {mean0:.6f} (29/12 = {29/12:.6f}), sd = {sd0:.4f}")

$P_0(T_0=2)=0$이 그대로 보인다 — 1에서 0으로 바로 가는 화살표가 없기 때문이다. 이런 '구멍'은 04a의 일반 재생 과정에서는 없던, 마르코프 구조가 주는 특징이다.

### 4.2 복귀시간 시뮬레이션 (Fig 1)

`simulate_chain`으로 0에서 출발하는 긴 경로 여러 개를 만들고, **모든 0-복귀시간을 한 자루에 모은다.** 정리 1이 이것들이 i.i.d.라고 보장하므로 그냥 `mc_estimate`를 써도 된다. i.i.d. 주장 자체는 연속한 복귀시간의 lag-1 자기상관으로 검사한다.

In [ ]:
X = simulate_chain(P, 0, N_STEPS, rng, n_paths=N_PATHS)      # shape (N_PATHS, N_STEPS+1)
rt0 = return_times(X, 0)
rt2 = return_times(X, 2)
est_rt0, est_rt2 = mc_estimate(rt0), mc_estimate(rt2)
p_rt0_eq3, p_rt0_eq2 = mc_proportion(rt0 == 3), mc_proportion(rt0 == 2)

# i.i.d. 검사: 경로별 lag-1 자기상관, 경로 평균 (SE ~ 1/sqrt(n_returns per path) / sqrt(N_PATHS))
ac1 = np.array([np.corrcoef(r[:-1], r[1:])[0, 1]
                for r in (np.diff(np.flatnonzero(row == 0)) for row in X)])
est_ac1 = mc_estimate(ac1)
print(f"returns to 0 pooled: n = {len(rt0)},  E_0[T_0] est = {est_rt0}   (exact {29/12:.4f})")
print(f"returns to 2 pooled: n = {len(rt2)},  E_2[T_2] est = {est_rt2}   (exact {29/8:.4f})")
print(f"P_0(T_0=3) est = {p_rt0_eq3}  (exact 0.25);  P_0(T_0=2) est = {p_rt0_eq2}  (exact 0)")
print(f"lag-1 autocorrelation of successive return times: {est_ac1}")

In [ ]:
fig, ax = plt.subplots()
K = 15
plot_hist_vs_pmf(rt0[rt0 <= K], k[1:K + 1], pmf0[1:K + 1], ax=ax, label_pmf="exact pmf (first-step)")
ax.set_xlabel("return time to state 0,  $T_0$"); ax.set_ylabel("probability")
ax.set_title(f"Fig 1. Return times to 0 pooled over {N_PATHS} paths (n = {len(rt0)})")
ax.text(0.97, 0.55, f"mean {est_rt0.mean:.4f} vs 29/12 = {29/12:.4f}\n"
        f"P(T_0 = 2) = {p_rt0_eq2.mean:.4f}\nlag-1 autocorr = {est_ac1.mean:+.4f} ± {est_ac1.se:.4f}",
        transform=ax.transAxes, ha="right", va="top", fontsize=9,
        bbox=dict(boxstyle="round", fc="white", alpha=0.8))
plt.show()

히스토그램은 $n=2$에서 비어 있고 정확 pmf의 막대와 겹친다. 자기상관은 0의 몇 SE 안이다 — 사이클끼리 독립이라는 정리 1의 시뮬레이션 버전.

### 4.3 재생-보상과 에르고딕 정리 (Fig 2)

같은 경로에서 이번엔 **사이클 안에서 번 것**을 센다: 0-사이클당 상태 1 방문 수 $V_1$(정리 2 (iii): $\pi(1)/\pi(0)=3/4$), 상태 2 방문 수 $V_2$($=2/3$). 그리고 경로 단위 시간비율과 $f(x)=x^2$의 시간평균(정리 3: $41/29$). 시간비율의 SE는 경로 반복으로 구한다(함정 5).

In [ ]:
V1 = np.concatenate([visits_per_cycle(X[i], 0, 1) for i in range(N_PATHS)])
V2 = np.concatenate([visits_per_cycle(X[i], 0, 2) for i in range(N_PATHS)])
est_V1, est_V2 = mc_estimate(V1), mc_estimate(V2)

frac = np.stack([(X[:, 1:] == y).mean(1) for y in range(3)], axis=1)   # 경로별 시간비율, shape (N_PATHS, 3)
frac2 = frac[:, 2]
f_avg = (X[:, 1:] ** 2).mean(1)                                        # 경로별 (1/n) sum X_k^2
est_frac2, est_favg = mc_estimate(frac2), mc_estimate(f_avg)

print(f"visits to 1 per 0-cycle: {est_V1}  (exact {expected_visits_exact(P, 0, 1):.4f} = pi(1)/pi(0))")
print(f"visits to 2 per 0-cycle: {est_V2}  (exact {expected_visits_exact(P, 0, 2):.4f} = pi(2)/pi(0))")
print("time fractions (mean over paths):", frac.mean(0).round(4), " vs pi =", pi.round(4))
print(f"fraction in 2: {est_frac2}  (exact {8/29:.5f});  mean of X^2: {est_favg}  (exact {41/29:.5f})")
print("empirical_distribution (all paths, burn_in=1):", empirical_distribution(X, 3, burn_in=1).round(4))

In [ ]:
n_axis = np.arange(1, N_STEPS + 1)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
for y in range(3):
    axL.plot(n_axis, running_mean(X[0, 1:] == y), color=f"C{y}", lw=1, label=f"state {y}")
    axL.axhline(pi[y], color=f"C{y}", ls="--", lw=1)
axL.set_xscale("log"); axL.set_xlabel("n (log)"); axL.set_ylabel("running fraction of time in state")
axL.set_title("Fig 2a. One path: N_n(y)/n -> pi(y) = (12, 9, 8)/29 (dashed)"); axL.legend()
axR.plot(n_axis, running_mean(X[0, 1:] ** 2), color="C3", lw=1, label="running mean of $X_n^2$")
axR.axhline(41 / 29, color="k", ls="--", lw=1, label="41/29")
axR.set_xscale("log"); axR.set_xlabel("n (log)"); axR.set_ylabel("$\\frac{1}{n}\\sum_{k \\leq n} X_k^2$")
axR.set_title("Fig 2b. Ergodic theorem, f(x) = x^2"); axR.legend()
plt.tight_layout(); plt.show()

한 경로의 시간비율은 $n$이 커지며 $\pi$로, $x^2$의 시간평균은 $41/29$로 간다. 정리 2·3은 "거의 모든 경로"에서 이렇게 된다고 말한다 — 경로 하나만 봐도 $\pi$를 알 수 있다는 뜻이다.

### 4.4 주기적 사슬: Ehrenfest(공 4개) (Fig 3)

01e의 수렴정리는 비주기성을 요구했다. 정리 2는 요구하지 않는다. 주기 2인 Ehrenfest 사슬에서 두 결과를 나란히 놓는다: $P^n(0,0)$은 진동하고, 시간비율은 $\pi(0)=1/16=1/E_0[T_0]$로 수렴한다.

In [ ]:
Pe = ehrenfest_matrix(4)
info_e = classify_states(Pe)
pi_e = stationary(Pe)
ET_e = np.array([mean_return_time_exact(Pe, x) for x in range(5)])
print("period:", info_e["period"], "| pi_e * 16 =", np.round(pi_e * 16, 10), "= C(4, k)")
print("E_x[T_x] first-step:", ET_e.round(6), "| 1/pi_e:", (1 / pi_e).round(6))
np.testing.assert_allclose(pi_e * ET_e, 1.0, atol=1e-10)

N_POWER_MAX = 40
n_pow = np.arange(1, N_POWER_MAX + 1)
Pn00 = np.array([np.linalg.matrix_power(Pe, n)[0, 0] for n in n_pow])
print("P^n(0,0), n=1..8:", Pn00[:8].round(4), "|  even-n limit 2*pi(0) =", 2 * pi_e[0])

In [ ]:
Xe = simulate_chain(Pe, 0, N_STEPS, rng, n_paths=N_PATHS)
rte0, rte2 = return_times(Xe, 0), return_times(Xe, 2)
frac0_e = (Xe[:, 1:] == 0).mean(1)
est_rte0, est_rte2, est_frac0e = mc_estimate(rte0), mc_estimate(rte2), mc_estimate(frac0_e)
print(f"Ehrenfest E_0[T_0]: {est_rte0}  (exact {ET_e[0]:.1f})")
print(f"Ehrenfest E_2[T_2]: {est_rte2}  (exact {ET_e[2]:.4f})")
print(f"Ehrenfest fraction of time at 0: {est_frac0e}  (exact {pi_e[0]:.4f})")

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
axL.bar(n_pow, Pn00, color="C0", width=0.8)
axL.axhline(2 * pi_e[0], color="C1", ls="--", lw=1, label="2 pi(0) = 1/8 (even n)")
axL.axhline(pi_e[0], color="k", ls=":", lw=1, label="pi(0) = 1/16")
axL.set_xlabel("n"); axL.set_ylabel("$P^n(0,0)$"); axL.legend()
axL.set_title("Fig 3a. Ehrenfest(4): $P^n(0,0)$ alternates, no limit")
axR.plot(n_axis, running_mean(Xe[0, 1:] == 0), color="C0", lw=1, label="running fraction at 0 (one path)")
axR.axhline(pi_e[0], color="k", ls="--", lw=1, label="pi(0) = 1/16 = 1/E_0[T_0]")
axR.set_xscale("log"); axR.set_xlabel("n (log)"); axR.set_ylabel("fraction of time at 0"); axR.legend()
axR.set_title("Fig 3b. ...but the time average converges")
plt.tight_layout(); plt.show()

왼쪽: 홀수 $n$에서 0, 짝수 $n$에서 $\to2\pi(0)$ — 04b의 상수 간격 재생 과정에서 $A(t)=t\bmod d$가 진동하던 것과 같은 현상. 오른쪽: 그래도 시간비율은 $1/16$. 진동을 평균하면 $\frac12(0+2\pi(0))=\pi(0)$이다 — 체사로 평균만 수렴한다.

### 4.5 양재귀를 깨면: $\mathbb Z$ 위 단순 대칭 랜덤워크 (Fig 4)

SRW는 재귀(01x)이지만 $E_0[T_0]=\infty$(영재귀)다. 정리 2의 1줄은 여전히 성립하여 $N_n(0)/n\to1/\infty=0$: 시간비율이 0으로 가고 정상분포는 없다. 참조값만 그린다(compare/assert 없음).

In [ ]:
N_RW_PATHS, N_RW_STEPS = scale(2000), 10_000
W = random_walk_paths(N_RW_PATHS, N_RW_STEPS, rng)
n_arr = np.unique(np.logspace(2, 4, 20).astype(int))
frac0_rw = np.array([(W[:, 1:n + 1] == 0).mean() for n in n_arr])
horizons = [100, 1000, 10_000]
obs_mean_rt = [return_times(W[:, :H + 1], 0).mean() for H in horizons]
for H, m in zip(horizons, obs_mean_rt):
    print(f"horizon {H:>6d}: mean of observed return times to 0 = {m:6.2f}")
print(f"fraction of time at 0 up to n = {N_RW_STEPS}: {frac0_rw[-1]:.5f}  (ref sqrt(2/(pi n)) = {np.sqrt(2/(np.pi*N_RW_STEPS)):.5f})")

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
axL.loglog(n_arr, frac0_rw, "o-", color="C0", label=f"simulation ({N_RW_PATHS} paths)")
axL.loglog(n_arr, np.sqrt(2 / (np.pi * n_arr)), "--", color="k", label="sqrt(2/(pi n)), slope -1/2")
axL.set_xlabel("n (log)"); axL.set_ylabel("fraction of time at 0 up to n (log)"); axL.legend()
axL.set_title("Fig 4a. SRW on Z: N_n(0)/n -> 0, so pi(0) = 0")
axR.bar([str(H) for H in horizons], obs_mean_rt, color="C1")
axR.set_xlabel("horizon H"); axR.set_ylabel("mean of return times observed by H")
axR.set_title("Fig 4b. Observed mean return time grows with H: E_0[T_0] = inf")
plt.tight_layout(); plt.show()

관측된 복귀시간의 평균은 지평이 길어질수록 커진다 — 유한 지평 시뮬레이션은 긴 복귀를 잘라내므로 항상 $E_0[T_0]$을 과소평가하고, 진짜 값은 $\infty$다. 04a의 재생 정리는 $\mu=\infty$일 때 $N(t)/t\to0$을 주며, 이것이 "$\pi(0)=0$"의 뜻이다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드에서 계산한다: `stationary`/`absorption`(정리 2 (ii)), 첫걸음 pmf, 기본행렬(정리 4), $\sum\pi f$(정리 3).

In [ ]:
exact = {
    "E0_T0": ET[0],                       # 첫걸음 분석 = 1/pi(0)
    "E2_T2": ET[2],
    "P0_T0_eq_3": pmf0[3],                # a Q R
    "visits_1_per_cycle_from_0": expected_visits_exact(P, 0, 1),   # = pi(1)/pi(0)
    "frac2": pi[2],
    "ergodic_x2": pi @ np.arange(3) ** 2,
    "ehrenfest_E0_T0": ET_e[0],
    "ehrenfest_frac0": pi_e[0],
    "ehrenfest_E2_T2": ET_e[2],
}
rows = [
    {"name": "E_0[T_0] (3-state chain)", "predicted": predictions["E0_T0"], "estimate": est_rt0, "exact": exact["E0_T0"]},
    {"name": "E_2[T_2]", "predicted": predictions["E2_T2"], "estimate": est_rt2, "exact": exact["E2_T2"]},
    {"name": "P_0(T_0 = 3)", "predicted": predictions["P0_T0_eq_3"], "estimate": p_rt0_eq3, "exact": exact["P0_T0_eq_3"]},
    {"name": "E_0[V_1] visits to 1 per 0-cycle", "predicted": predictions["visits_1_per_cycle_from_0"], "estimate": est_V1, "exact": exact["visits_1_per_cycle_from_0"]},
    {"name": "long-run fraction of time in state 2", "predicted": None, "estimate": est_frac2, "exact": exact["frac2"]},
    {"name": "ergodic average of X_n^2", "predicted": predictions["ergodic_x2"], "estimate": est_favg, "exact": exact["ergodic_x2"]},
    {"name": "Ehrenfest(4) E_0[T_0]", "predicted": predictions["ehrenfest_E0_T0"], "estimate": est_rte0, "exact": exact["ehrenfest_E0_T0"]},
    {"name": "Ehrenfest(4) fraction of time at 0", "predicted": None, "estimate": est_frac0e, "exact": exact["ehrenfest_frac0"]},
    {"name": "Ehrenfest(4) E_2[T_2]", "predicted": None, "estimate": est_rte2, "exact": exact["ehrenfest_E2_T2"]},
]
Markdown(compare_table(rows))

각 행이 어긋난다면 무엇이 의심되는가:
- **E_0[T_0], E_2[T_2]**: 첫걸음 분석(`absorption`)과 `stationary`가 다른 사슬을 보고 있거나, `return_times`가 첫 도달 이전 조각을 포함했다.
- **P_0(T_0 = 3)**: $aQ^{n-2}R$ 인덱싱(시각 1 자기고리 처리)이 틀렸다.
- **E_0[V_1]**: 사이클 경계 규약(시각 0 포함, 복귀 시각 제외)이 `visits_per_cycle`과 기본행렬 계산에서 서로 다르다.
- **fraction in 2, X_n^2 평균**: 경로 길이가 짧아 초기 조각(burn-in)이 남았거나, 경로 단위 SE 대신 시각 단위 SE를 썼다(함정 5).
- **Ehrenfest 세 행**: 주기성이 시간평균을 깨뜨렸다면 정리 2가 틀린 것 — 실제로는 안 깨진다. 어긋나면 `return_times`/`frac`의 버그다.

In [ ]:
assert_close(est_rt0, exact["E0_T0"], k=4, name="E_0[T_0]")
assert_close(est_rt2, exact["E2_T2"], k=4, name="E_2[T_2]")
assert_close(p_rt0_eq3, exact["P0_T0_eq_3"], k=4, name="P_0(T_0=3)")
assert_close(est_V1, exact["visits_1_per_cycle_from_0"], k=4, name="visits to 1 per 0-cycle")
assert_close(est_frac2, exact["frac2"], k=4, name="fraction of time in 2")
assert_close(est_favg, exact["ergodic_x2"], k=4, name="ergodic average of X^2")
assert_close(est_rte0, exact["ehrenfest_E0_T0"], k=4, name="Ehrenfest E_0[T_0]")
assert_close(est_frac0e, exact["ehrenfest_frac0"], k=4, name="Ehrenfest fraction at 0")
assert_close(est_rte2, exact["ehrenfest_E2_T2"], k=4, name="Ehrenfest E_2[T_2]")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

$\{0,1,2\}$ 위의 반사 걸음 $P(0,1)=1$, $P(1,0)=P(1,2)=\frac12$, $P(2,1)=1$.
(a) $\pi$를 구하라. (b) 첫걸음 분석으로 $E_0[T_0]$, $E_1[T_1]$을 손으로 구하고 $1/\pi$와 비교하라. (c) 이 사슬의 주기는? $P^n(0,0)$은 수렴하는가? 시간비율은?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\pi P=\pi$: $\pi(0)=\frac12\pi(1)$, $\pi(2)=\frac12\pi(1)$, 합 1 → $\pi=(\frac14,\frac12,\frac14)$.

(b) $h_y=E_y[T_0]$: $h_1=1+\frac12h_2$, $h_2=1+h_1$ → $h_1=3$, $h_2=4$. $E_0T_0=1+h_1=4=1/\pi(0)$ ✓. $E_1T_1=1+\frac12E_0[T_1]+\frac12E_2[T_1]=1+\frac12\cdot1+\frac12\cdot1=2=1/\pi(1)$ ✓. 대칭으로 $E_2T_2=4$.

(c) 주기 2 (0에서 돌아오려면 항상 짝수 걸음). $P^n(0,0)$은 홀수 $n$에서 0이라 수렴하지 않지만 시간비율은 $\frac14$ — 정리 2는 비주기성이 불필요하다.

```python
Pr = np.array([[0., 1., 0.], [.5, 0., .5], [0., 1., 0.]])
print(stationary(Pr), classify_states(Pr)["period"])                  # [0.25 0.5 0.25] {0: 2}
print([mean_return_time_exact(Pr, x) for x in range(3)])               # [4.0, 2.0, 4.0]
print([np.linalg.matrix_power(Pr, n)[0, 0] for n in range(1, 7)])      # 0, .5, 0, .5, ...
Xr = simulate_chain(Pr, 0, 20_000, rng, n_paths=20)
print(mc_estimate((Xr[:, 1:] == 0).mean(1)))                           # ≈ 0.25
```

</details>

### E2 유도 후 시뮬레이션 검증

(a) 재생-보상 정리로 $\pi(y)/\pi(x)=E_x[V_y]$를 유도하라(정리 2 (iii)). (b) $x$를 흡수 상태로 만든 기본행렬 $N=(I-Q)^{-1}$로 $E_x[V_y]=\sum_{z\ne x}P(x,z)N_{zy}$임을 설명하고 `absorption(P_x, others, [x])['N']`으로 $E_2[V_0]$을 정확히 계산하라. (c) 4.2의 경로 `X`에서 2-사이클당 0 방문 수를 세어 4 SE 안에 들어오는지 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 보상 $R_k=$ 사이클 $k$의 $y$ 방문 수. $(T^{(k)}_x-T^{(k-1)}_x,R_k)$는 i.i.d.(정리 1), $0\le R_k\le$ 사이클 길이라 적분가능. 재생-보상으로 $N_n(y)/n\to E_x[V_y]/E_x[T_x]$. 이 극한이 $\pi(y)$(정리 2 증명 3줄)이고 $1/E_x[T_x]=\pi(x)$이므로 $E_x[V_y]=\pi(y)/\pi(x)$.

(b) 시각 1에 $z$에 있을 확률은 $P(x,z)$. $z\ne x$에서 출발해 $x$에 흡수되기 전까지 $y$를 방문하는 기대 횟수가 기본행렬의 $N_{zy}$(01b). $V_y$는 시각 0의 $x$를 세지 않으므로 $y\ne x$이면 이대로 $E_x[V_y]=\sum_{z\ne x}P(x,z)N_{zy}$. $z=x$ 항은 $V_y$에 기여하지 않는다(사이클이 시각 1에 끝남). $E_2[V_0]=\pi(0)/\pi(2)=12/8=1.5$.

(c) 시뮬레이션 $\approx1.50\pm0.002$(full) 로 4 SE 안.

```python
E2V0 = expected_visits_exact(P, 2, 0)                     # 1.5
V0_from2 = np.concatenate([visits_per_cycle(X[i], 2, 0) for i in range(N_PATHS)])
est = mc_estimate(V0_from2)
print(E2V0, pi[0] / pi[2], est)
assert_close(est, E2V0, k=4, name="E_2[V_0]")
# 참고: E_0[V_1] = 0.75, E_0[V_2] = 2/3 (4.3에서 확인함)
```

</details>

### E3 가정을 깨보기

$\mathbb Z$ 위 단순 대칭 랜덤워크(01x: 재귀). (a) $P_0(T_0>2m)=\binom{2m}m4^{-m}\sim1/\sqrt{\pi m}$을 이용해 $E_0[T_0]=\sum_{n\ge0}P_0(T_0>n)=\infty$임을 보여라. (b) `random_walk_paths`로 시간의 몇 %를 0에서 보내는지, 그리고 지평 100/1000/10000까지 관측된 복귀시간의 평균을 구하라(4.5의 `W` 재사용 가능). (c) 정리 2에서 무엇이 깨졌고 $\pi(0)$은 얼마인가? 정리 4의 정상 측도는 무엇인가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 꼬리합 공식 $E[T]=\sum_{n\ge0}P(T>n)$에서 $P_0(T_0>2m)=P_0(T_0>2m+1)$이고 Stirling으로 $\binom{2m}m4^{-m}\sim1/\sqrt{\pi m}$이므로 $E_0[T_0]\ge\sum_m1/\sqrt{\pi m}=\infty$.

(b) $E[\#\{1\le k\le n:S_k=0\}]=\sum_{1\le j\le n/2}\binom{2j}j4^{-j}\approx\sqrt{2n/\pi}$: $n=10^4$에서 $79.8$회, 비율 $\approx0.008$. 관측 복귀시간 평균은 지평 100/1000/10000에서 약 7, 20, 60 — 지평과 함께 발산. $P_0(T_0>100)=\binom{100}{50}4^{-50}\approx0.0796$이므로 8%의 사이클은 지평 100 안에 끝나지도 않는다.

(c) **양재귀**가 깨졌다(영재귀). 정리 2의 1줄은 살아남아 $N_n(0)/n\to0=1/\infty$, 따라서 $\pi(0)=0$이고 대칭으로 모든 상태가 0 → 확률분포가 될 수 없다: 정상분포 없음. 정리 4의 $\mu_0(y)=E_0[V_y]=1$(모든 $y$; 계수측도)은 $\mu_0P=\mu_0$를 만족하는 정상 측도이지만 총질량 $\infty$라 정규화가 안 된다.

```python
from math import comb
n = N_RW_STEPS
exact_visits = sum(comb(2*j, j) / 4**j for j in range(1, n // 2 + 1))
print(exact_visits, np.sqrt(2 * n / np.pi))                        # 78.79 vs 79.79 (시각 0 제외/포함 차이 1)
print(comb(100, 50) / 4**50)                                        # P_0(T_0 > 100) = 0.0796
print((W[:, 1:] == 0).sum(1).mean(), (W[:, 1:] == 0).mean())       # ≈ 78.8 visits, fraction ≈ 0.0079
print([return_times(W[:, :H + 1], 0).mean() for H in (100, 1000, 10_000)])   # ≈ 7, 20, 60
```

</details>

## 7. 정리

1. 상태 $x$로의 복귀 시각들은 강마르코프 성질에 의해 재생 과정이고, 사이클(경로 조각)은 i.i.d.다.
2. 재생 SLLN → $N_n(x)/n\to1/E_x[T_x]$; 재생-보상 + 정상 출발 + 유계수렴 → $\pi(x)=1/E_x[T_x]$, $\pi(y)/\pi(x)=E_x[V_y]$. 기약·양재귀만 필요, 비주기성 불필요.
3. 예제: $\pi=(12,9,8)/29$, $E_0T_0=29/12$, $E_2T_2=29/8$, 0-사이클당 1 방문 $0.75$; Ehrenfest(4): $E_0T_0=16$이지만 $P^n(0,0)$은 진동.
4. 에르고딕 정리 $\frac1n\sum f(X_k)\to\sum\pi f$는 같은 재생-보상 논법; $f=x^2$에서 $41/29$.
5. 영재귀(SRW on $\mathbb Z$)에서는 $E_0T_0=\infty$, $\pi(0)=0$: 재귀는 정상분포의 존재를 보장하지 않는다. 01d↔04 연결: 정상분포 = 시간비율 = 1/평균 복귀시간.

**Trap 카드**
- Q: 주기가 2인 기약 유한 사슬(예: Ehrenfest)에는 정상분포 $\pi$가 있는가? $\pi(x)=1/E_x[T_x]$는 성립하는가? $P^n(x,x)\to\pi(x)$는?
- A: 있다(유한 기약이면 유일하게 존재). 성립한다(비주기성은 이 항등식과 시간평균에 필요 없다). 성립하지 않는다: $P^n(0,0)$은 홀수 $n$에서 0, 짝수 $n$에서 $\to2\pi(0)$으로 진동하며 체사로 평균만 $\pi(0)$으로 간다.

**다음 노트북: 05a** — 재생 사이클이 '기억이 리셋되는 정지시간'으로 잘리는 것을 보았다. 다음 모듈은 정지시간과 조건부 기대를 일반화하는 마팅게일이다: 05a에서 조건부 기대의 성질과 마팅게일 동물원($S_n^2-n$, $Z_n/m^n$ 등)을 다루고, 05b의 선택적 정지 정리는 04a의 Wald 항등식을 포함하는 더 큰 그림이다.

log/progress.md 한 줄 템플릿:
`- [ ] 04c 재생 관점의 마르코프 사슬 — YYYY-MM-DD, 예측 6개 중 __개 적중, 막힌 곳: ____, 다시 볼 것: 정리 2 증명 3줄의 가정 표시`